# Muon Regime Scaling Experiment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/colab/vit_optimizer_diagnostics/muon_regime_scaling/02_muon_regime_scaling.ipynb)

첫 번째 notebook에서 old-success regime 복원이 확인되면, 이번에는 Muon과 진단량을 고정하고 외부 제어변수를 한 축씩 바꾼다.

핵심 관점은 `diagnostic_i = diagnostic_i(N, B, update_budget, seed | model, optimizer fixed)`이다.

## 0. 실험 그룹

- `regime_endpoints`: 현재 실패 조건과 과거 성공 조건을 각각 재현
- `data_scale_fixed_updates`: batch=512, seed=7, 총 update 수를 약 3900으로 맞추고 N만 10k→20k→40k
- `update_budget_at_40k`: N=40k, batch=512, seed=7에서 update budget만 증가

한 번에 여러 full run을 돌리면 오래 걸리므로 한 그룹씩 실행한다.

In [ ]:
!pip -q install datasets prodigyopt tensorboard scikit-learn

%cd /content
!rm -rf deep-learning-diagnostics-and-improvement
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git
%cd /content/deep-learning-diagnostics-and-improvement/colab/vit_optimizer_diagnostics/muon_regime_scaling

import sys
import math
from pathlib import Path

PARENT = Path.cwd().parent
sys.path.insert(0, str(PARENT))
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch

from muon_regime_runner import run_muon_regime

## 1. 실행할 scaling 축 선택

기본값은 가장 해석이 단순한 `update_budget_at_40k`이다. N/B/seed를 고정하고 optimization time만 바꾼다.

In [ ]:
EXPERIMENT_GROUP = "update_budget_at_40k"
# 다른 선택:
# EXPERIMENT_GROUP = "data_scale_fixed_updates"
# EXPERIMENT_GROUP = "regime_endpoints"

TARGET_UPDATES = 3900

def epochs_for_updates(train_samples, batch_size, target_updates):
    steps_per_epoch = math.ceil(train_samples / batch_size)
    return math.ceil(target_updates / steps_per_epoch)

GROUPS = {
    "regime_endpoints": [
        {
            "name": "current_failed_regime_replay",
            "seed": 42,
            "train_samples": 10_000,
            "batch_size": 256,
            "epochs": 50,
            "validation_mode": "current_test_2k",
        },
        {
            "name": "old_success_regime_replay",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": 50,
            "validation_mode": "old_train_holdout_5k",
        },
    ],
    "data_scale_fixed_updates": [
        {
            "name": "N10k_B512_U3900",
            "seed": 7,
            "train_samples": 10_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(10_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N20k_B512_U3900",
            "seed": 7,
            "train_samples": 20_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(20_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U3900",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
    ],
    "update_budget_at_40k": [
        {
            "name": "N40k_B512_U1000",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 1000),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U2000",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 2000),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U3900",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 3900),
            "validation_mode": "old_train_holdout_5k",
        },
    ],
}

CONFIGS = GROUPS[EXPERIMENT_GROUP]
pd.DataFrame(CONFIGS)

## 2. 선택한 그룹 실행

각 configuration은 별도 폴더를 사용하지만 내부 `run_name`은 `muon`으로 유지한다. 그래서 optimizer 구현을 바꾸지 않고 동일 Muon을 반복 측정한다.

주의: 각 run 뒤에 representation/Hessian/Jacobian/tangent/manifold/trajectory까지 수행하므로 학습만 하는 sweep보다 오래 걸린다.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path("/content/muon_regime_scaling_outputs") / EXPERIMENT_GROUP

results = {}
summaries = []

for config in CONFIGS:
    print("=" * 80)
    print("running:", config["name"])
    print(config)

    result = run_muon_regime(
        config=config,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
        dynamics_every=10,
    )

    results[config["name"]] = result
    summaries.append(result["summary"])

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(OUTPUT_ROOT / "group_summary.csv", index=False)
summary_df

## 3. 핵심 질서변수 표

최종 accuracy 하나가 아니라 어느 scale에서 내부 상태가 함께 바뀌는지 본다.

In [ ]:
core_columns = [
    "name",
    "seed",
    "train_samples",
    "batch_size",
    "epochs",
    "approx_updates",
    "train_accuracy",
    "val_accuracy",
    "penultimate_cka_to_init",
    "penultimate_linear_probe",
    "nc1",
    "knn_purity",
    "margin_mean",
    "jacobian_spectral_norm",
    "jacobian_participation_rank",
    "tangent_target_alignment",
    "hessian_min_ritz",
    "hessian_max_ritz",
    "relative_sharpness",
    "path_to_chord_ratio",
]

summary_df[core_columns]

## 4. 현재 대조군까지 붙이기

과거 성공 / 현재 실패의 저장된 reference와 이번 scaling 결과를 한 표에 합친다.

In [ ]:
reference = pd.read_csv("reference_results.csv")

scaling_for_compare = summary_df.rename(columns={"name": "run_id"})
comparison = pd.concat(
    [reference, scaling_for_compare],
    ignore_index=True,
    sort=False,
)

comparison.to_csv(OUTPUT_ROOT / "reference_plus_scaling.csv", index=False)
comparison

## 5. 해석 순서

1. **학습 가능성** — train/val accuracy가 어느 scale에서 회복되는가?
2. **feature movement** — CKA/effective rank가 함께 바뀌는가?
3. **task organization** — probe/kNN 상승, NC1 하락, margin 양수화가 일어나는가?
4. **function sensitivity** — Jacobian spectral norm과 participation rank가 안정화되는가?
5. **tangent geometry** — target alignment와 tangent effective rank가 변하는가?
6. **local loss geometry** — Hessian의 극단적 ± Ritz와 relative sharpness가 완화되는가?
7. **optimization path** — path/chord ratio가 어느 regime에서 달라지는가?

여러 진단량이 accuracy 회복 지점 근처에서 함께 방향을 바꾸면 단순 상관보다 훨씬 강한 regime-transition 증거가 된다.

## 6. 실험 설계상 주의

`data_scale_fixed_updates`는 update 수를 맞추지만 작은 N에서는 같은 샘플을 더 많이 반복해서 본다. 따라서 N 효과와 optimization-time을 분리하는 실험이지, information exposure까지 완전히 동일하게 만드는 실험은 아니다.

`update_budget_at_40k`가 가장 해석이 단순하므로 먼저 실행하는 것을 권장한다.

`empirical_dichotomy_capacity`는 현재 관측에서 floor에 가까웠으므로 그대로 기록하되 핵심 판정은 radius/dimension/axis geometry 쪽에 둔다.